# L34 · PPO 基础：强化学习直觉

**学习目标**
- 理解「强化学习（RL）」：靠奖励信号自学，而非靠标注答案
- 理解 PPO 的核心直觉：试错 → 拿到奖励 → 强化好动作
- 用 numpy 让一个 agent 学会「朝高奖励方向走」，看策略变强

**前置依赖**：L31（SFT）、L32（奖励）、L10（概率/期望）  
**预计时长**：55 分钟  
**技术栈**：`numpy`、`matplotlib`（离线可运行）

---

## 概念讲解：强化学习 = 训小狗用零食

监督学习是「给标准答案」；强化学习是「做对了给零食（奖励），做错了没有」。
小狗（agent）自己摸索：哪动作带来零食，就多干。

**PPO（Proximal Policy Optimization）** 是当今最强 RL 算法之一，ChatGPT 的 RLHF 就用它。
核心直觉：「小幅、稳定」地调整策略，别一次改太猛导致崩掉（和 DPO 的 β 约束同源思想）。

## 第一步：定义一个「环境」—— 朝右走有奖励

In [ ]:
import numpy as np
np.random.seed(3)

def reward(x):
    # 位置越靠右（x 越大）奖励越高；这就是环境给的「零食」
    return x

# 策略：根据当前状态输出「动作方向」的参数（越小越往左，越大越往右）
policy = np.array([0.0])     # 初始：不偏不倚
print("初始策略参数：", policy, "（agent 还不知道该往哪走）")

## 第二步：PPO 风格的「试错-强化」循环

In [ ]:
def sigmoid(x): return 1 / (1 + np.exp(-x))
lr = 0.05
progress = []
for episode in range(200):
    # 1. agent 按策略选动作（加噪声探索）
    action = sigmoid(policy[0]) * 2 - 1 + np.random.normal(0, 0.3)
    x = action * 2                         # 动作决定位置
    r = reward(x)                         # 2. 环境给奖励
    # 3. 强化：奖励高 → 朝增大该动作的方向更新（策略梯度）
    policy[0] += lr * r * (action) * 0.5
    if episode % 20 == 0:
        progress.append(policy[0])
print(f"训练后策略参数：{policy[0]:.3f}（应明显 > 0，说明学会往右走）")

# 🎯 AHA 顿悟单元格：看 agent 自己「学会」最优策略

运行下面代码。你会看到一张**策略强度上升曲线**：agent 一开始乱走（奖励低），
随着 PPO 试错-强化，它逐渐发现「往右走=零食多」，策略参数稳定偏向正方向，平均奖励持续攀升。

> 你刚实现的，就是强化学习最小内核。AlphaGo、ChatGPT、机器人控制，本质都是这个循环：
> **试错 → 拿奖励 → 强化好动作**。只不过它们的「环境」是围棋/人类偏好/物理世界。

In [ ]:
# ===== 运行我！看 agent 通过 RL 变强 =====
import matplotlib.pyplot as plt
np.random.seed(3)
policy = np.array([0.0])
rewards_history = []
for episode in range(300):
    action = sigmoid(policy[0]) * 2 - 1 + np.random.normal(0, 0.3)
    x = action * 2
    r = reward(x)
    policy[0] += lr * r * action * 0.5
    rewards_history.append(r)

smooth = np.convolve(rewards_history, np.ones(20)/20, mode="valid")
plt.figure(figsize=(7, 4))
plt.plot(smooth, color="#2ca02c", lw=2)
plt.title("强化学习：agent 的平均奖励随训练攀升")
plt.xlabel("训练回合"); plt.ylabel("平均奖励")
plt.grid(alpha=0.3); plt.show()
print(f"  🤖 最终策略参数 {policy[0]:.3f}（>0 表示学会往右走拿高奖励）")
print("  ✨ 没有一条标注答案，agent 靠奖励信号自己学会了最优策略！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：RL 与监督学习的根本区别（无标注、靠奖励）；策略梯度直觉「好动作多干」。  
**易错点**：探索噪声方差；更新公式符号（奖励×动作方向推动参数）。本演示为「Reward-weighted」简化版，非严格 PPO clip，但传达核心直觉。  
**AHA 机制**：奖励上升曲线，强「无师自通」震撼，呼应 L10 期望/概率。  
**衔接**：L35 RLHF（把人类偏好当奖励喂给 PPO）；L36 RL 项目（更复杂环境）。  
**依赖**：`pip install numpy matplotlib`。  
**真 PPO 说明**：真实 PPO 有 clip 机制、 critic(价值网络)、GAE 优势估计，需说明本课是直觉投影。

# 📚 作业 / 下一步

1. 把 `reward(x)` 改成 `-abs(x-3)`（奖励在 x=3 处最高），看 agent 是否学会停在中间。
2. 搜索「PPO clip」「advantage」了解真实 PPO 的稳键技巧。
3. 下一课 **L35 RLHF 全流程：从人类反馈到策略优化** —— 把 L31-L34 串成一条完整对齐流水线。